In [1]:
import numpy as np
import pandas as pd
from functools import lru_cache


# ------------------------------------------------------------------
# Load decomposition
# ------------------------------------------------------------------

artifact = np.load("../ml_artifacts/deck_ppmi_svd.npz")

U = artifact["U"]
sigma = artifact["sigma"]
card_ids = artifact["card_ids"]
card_names = artifact["card_names"].astype(str)

name_to_idx = {
    name.lower(): i
    for i, name in enumerate(card_names)
}


# ------------------------------------------------------------------
# Embedding construction
# ------------------------------------------------------------------

@lru_cache(maxsize=None)
def get_embedding(alpha=0.5):
    """
    Construct a card embedding:

        E = U @ Sigma^alpha

    alpha = 0.0  -> U
        Treats latent dimensions more equally.
        Useful for finding subtler structural relationships.

    alpha = 0.5  -> U Sigma^(1/2)
        Balanced representation. Good general default.

    alpha = 1.0  -> U Sigma
        Stronger emphasis on the dominant deck/archetype structure.

    Rows are L2-normalized so dot products are cosine similarities.
    """
    E = U * (sigma ** alpha)

    norms = np.linalg.norm(E, axis=1, keepdims=True)
    norms[norms == 0] = 1

    return E / norms


def _idx(card_name):
    try:
        return name_to_idx[card_name.lower()]
    except KeyError:
        raise ValueError(f"Unknown card: {card_name}")


def _centroid(card_list, E):
    indices = [_idx(card) for card in card_list]

    centroid = E[indices].mean(axis=0)

    norm = np.linalg.norm(centroid)
    if norm == 0:
        raise ValueError("Group has zero-length centroid")

    return centroid / norm


# ------------------------------------------------------------------
# 1. What cards occupy similar deck-building space?
# ------------------------------------------------------------------

def similar_cards(
    card_name,
    n=15,
    alpha=0.5,
):
    """
    Find cards that occur in similar deck-building contexts.

    Try changing alpha to see whether the relationship is driven by
    broad archetype structure or more specific latent structure.
    """
    E = get_embedding(alpha)

    i = _idx(card_name)

    scores = E @ E[i]
    scores[i] = -np.inf

    best = np.argsort(scores)[::-1][:n]

    return pd.DataFrame({
        "card": card_names[best],
        "similarity": scores[best],
    })


# ------------------------------------------------------------------
# 2. What cards best complete this package / shell?
# ------------------------------------------------------------------

def package_candidates(
    cards,
    n=20,
    alpha=0.5,
):
    """
    Treat a group of cards as a package and rank cards by similarity
    to the package as a whole.

    Examples:
        ["Underworld Breach", "Brain Freeze"]
        ["Lightning Bolt", "Chain Lightning", "Goblin Guide"]

    This is different from asking for neighbours of any one card:
    candidates need to agree with the combined latent context.
    """
    E = get_embedding(alpha)

    centroid = _centroid(cards, E)
    scores = E @ centroid

    used = {_idx(card) for card in cards}
    for i in used:
        scores[i] = -np.inf

    best = np.argsort(scores)[::-1][:n]

    return pd.DataFrame({
        "card": card_names[best],
        "package_fit": scores[best],
    })


# ------------------------------------------------------------------
# 3. What cards bridge two different packages / archetypes?
# ------------------------------------------------------------------

def bridge_cards(
    group_a,
    group_b,
    n=20,
    alpha=0.5,
):
    """
    Find cards that fit BOTH groups.

    Rather than averaging the two groups together, the score is the
    minimum similarity to either group:

        bridge_score = min(similarity_to_A, similarity_to_B)

    This rewards genuine bridges instead of cards that are extremely
    good for one side and irrelevant to the other.

    Potential uses:
      - cards connecting two archetypes
      - overlap between two combo packages
      - cards supporting a pivot during a draft
      - cards that make a hybrid deck coherent
    """
    E = get_embedding(alpha)

    centroid_a = _centroid(group_a, E)
    centroid_b = _centroid(group_b, E)

    sim_a = E @ centroid_a
    sim_b = E @ centroid_b

    bridge_score = np.minimum(sim_a, sim_b)

    used = {
        _idx(card)
        for card in (list(group_a) + list(group_b))
    }

    for i in used:
        bridge_score[i] = -np.inf

    best = np.argsort(bridge_score)[::-1][:n]

    return pd.DataFrame({
        "card": card_names[best],
        "group_a_fit": sim_a[best],
        "group_b_fit": sim_b[best],
        "bridge_score": bridge_score[best],
    })

In [2]:
similar_cards(
    "Underworld Breach",
    alpha=0.0,
)

,card,similarity
0,Brain Freeze,0.796979
1,Lion's Eye Diamond,0.725608
2,Lotus Petal,0.438478
3,Tendrils of Agony,0.434910
4,Yawgmoth's Will,0.386746
5,Thundertrap Trainer,0.356610
6,Expressive Iteration,0.343588
7,Frantic Search,0.339087
8,Manamorphose,0.331559
9,Wheel of Fortune,0.306976


In [10]:
import duckdb
import numpy as np
import pandas as pd


DB_PATH = "../data/17lands.duckdb"


# ------------------------------------------------------------------
# Load real deck builds
# ------------------------------------------------------------------

def load_decks(
    db_path=DB_PATH,
    min_cards=8,
):
    """
    Return actual deck builds as lists of distinct card names.

    Basic lands are excluded because they dominate many deck contexts
    without being especially informative for this evaluation.
    """
    with duckdb.connect(db_path, read_only=True) as con:
        rows = con.execute(
            """
            SELECT
                dbc.build_id,
                list(c.card_name ORDER BY c.card_name) AS cards
            FROM deck_build_cards dbc
            JOIN cards c
                ON c.card_id = dbc.card_id
            WHERE dbc.deck_count > 0
              AND c.card_name NOT IN (
                    'Plains',
                    'Island',
                    'Swamp',
                    'Mountain',
                    'Forest',
                    'Wastes'
              )
            GROUP BY dbc.build_id
            HAVING count(*) >= ?
            """,
            [min_cards],
        ).fetchall()

    decks = []

    for _, cards in rows:
        cards = [
            card
            for card in cards
            if card.lower() in name_to_idx
        ]

        if len(cards) >= min_cards:
            decks.append(cards)

    return decks


decks = load_decks()

print(f"Loaded {len(decks):,} deck builds")
print("Example:", decks[0][:10])

Loaded 12,441 deck builds
Example: ['Black Lotus', 'Booster Tutor', 'Brain Freeze', 'Concealing Curtains', 'Consult the Star Charts', 'Counterspell', 'Dismember', 'Fiery Islet', 'Flame Slash', "Jace, Vryn's Prodigy"]


In [13]:
def leave_one_out_test(
    decks,
    *,
    alpha=0.5,
    n_tests=1000,
    seed=42,
):
    """
    Remove one card from a real deck.

    Use the remaining cards to construct a context vector and rank every
    card in the embedding space.

    Good embeddings should rank the held-out card highly.
    """
    rng = np.random.default_rng(seed)
    E = get_embedding(alpha)

    results = []
    tests = []

    for deck in decks:
        if len(deck) < 2:
            continue

        target = rng.choice(deck)

        context = [
            card
            for card in deck
            if card != target
        ]

        if not context:
            continue

        tests.append(
            (context, target)
        )

    if len(tests) > n_tests:
        indices = rng.choice(
            len(tests),
            size=n_tests,
            replace=False,
        )

        tests = [
            tests[i]
            for i in indices
        ]

    for context, target in tests:
        context_idx = [
            name_to_idx[c.lower()]
            for c in context
        ]

        target_idx = name_to_idx[
            target.lower()
        ]

        centroid = E[
            context_idx
        ].mean(axis=0)

        norm = np.linalg.norm(
            centroid
        )

        if norm == 0:
            continue

        centroid /= norm

        scores = E @ centroid

        # Do not allow cards already present in the deck
        # to compete with the held-out card.
        scores[
            context_idx
        ] = -np.inf

        target_score = scores[
            target_idx
        ]

        rank = (
            np.sum(
                scores > target_score
            )
            + 1
        )

        results.append({
            "target": target,
            "rank": rank,
            "reciprocal_rank": 1 / rank,
            "hit_5": rank <= 5,
            "hit_10": rank <= 10,
            "hit_20": rank <= 20,
            "hit_50": rank <= 50,
        })

    df = pd.DataFrame(
        results
    )

    summary = pd.Series({
        "tests": len(df),
        "MRR": df[
            "reciprocal_rank"
        ].mean(),
        "median_rank": df[
            "rank"
        ].median(),
        "mean_rank": df[
            "rank"
        ].mean(),
        "hit@5": df[
            "hit_5"
        ].mean(),
        "hit@10": df[
            "hit_10"
        ].mean(),
        "hit@20": df[
            "hit_20"
        ].mean(),
        "hit@50": df[
            "hit_50"
        ].mean(),
    })

    return summary, df

In [14]:
summary, reconstruction_results = leave_one_out_test(
    decks,
    alpha=0.5,
    n_tests=2000,
)

summary

tests          2000.000000
MRR               0.061246
median_rank      59.000000
mean_rank        89.983000
hit@5             0.071000
hit@10            0.123500
hit@20            0.227000
hit@50            0.449000
dtype: float64

In [15]:
def compare_alphas(
    decks,
    alphas=(0.0, 0.25, 0.5, 0.75, 1.0),
    n_tests=1000,
):
    rows = []

    for alpha in alphas:
        summary, _ = leave_one_out_test(
            decks,
            alpha=alpha,
            n_tests=n_tests,
            seed=42,
        )

        rows.append({
            "alpha": alpha,
            **summary.to_dict(),
        })

    return pd.DataFrame(rows).set_index("alpha")


alpha_results = compare_alphas(
    decks,
    n_tests=2000,
)

alpha_results

,tests,MRR,median_rank,mean_rank,hit@5,hit@10,hit@20,hit@50
alpha,,,,,,,,
0.00,2000.0,0.054404,63.0,104.9115,0.0610,0.1140,0.2115,0.4315
0.25,2000.0,0.057604,60.0,90.7415,0.0665,0.1250,0.2205,0.4470
0.50,2000.0,0.061246,59.0,89.9830,0.0710,0.1235,0.2270,0.4490
0.75,2000.0,0.056891,62.0,92.5110,0.0660,0.1250,0.2185,0.4435
1.00,2000.0,0.054129,63.0,96.1015,0.0630,0.1205,0.2105,0.4330


In [16]:
def pair_discrimination_test(
    decks,
    *,
    alpha=0.5,
    n_pairs=10000,
    seed=42,
):
    """
    Compare similarity for:

      positive = two cards from the same real deck
      negative = two random cards

    Returns the probability that a random positive pair scores above
    a random negative pair.

    0.5 = random
    1.0 = perfect separation
    """
    rng = np.random.default_rng(seed)
    E = get_embedding(alpha)

    positive_scores = []
    negative_scores = []

    usable_decks = [
        deck
        for deck in decks
        if len(deck) >= 2
    ]

    for _ in range(n_pairs):
        # Positive pair
        deck = usable_decks[
            rng.integers(len(usable_decks))
        ]

        a, b = rng.choice(
            deck,
            size=2,
            replace=False,
        )

        ia = name_to_idx[a.lower()]
        ib = name_to_idx[b.lower()]

        positive_scores.append(
            E[ia] @ E[ib]
        )

        # Random negative pair
        i, j = rng.choice(
            len(card_names),
            size=2,
            replace=False,
        )

        negative_scores.append(
            E[i] @ E[j]
        )

    positive_scores = np.asarray(
        positive_scores
    )

    negative_scores = np.asarray(
        negative_scores
    )

    discrimination = np.mean(
        positive_scores
        > negative_scores
    )

    return pd.Series({
        "pair_discrimination": discrimination,
        "positive_mean": positive_scores.mean(),
        "negative_mean": negative_scores.mean(),
        "positive_median": np.median(positive_scores),
        "negative_median": np.median(negative_scores),
    })

In [17]:
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    result = pair_discrimination_test(
        decks,
        alpha=alpha,
    )

    print()
    print(f"alpha = {alpha}")
    print(result)


alpha = 0.0
pair_discrimination    0.593200
positive_mean          0.103521
negative_mean          0.026645
positive_median        0.043703
negative_median        0.008879
dtype: float64

alpha = 0.25
pair_discrimination    0.658500
positive_mean          0.194138
negative_mean          0.063258
positive_median        0.096962
negative_median        0.018199
dtype: float64

alpha = 0.5
pair_discrimination    0.690200
positive_mean          0.307033
negative_mean          0.122983
positive_median        0.209827
negative_median        0.022888
dtype: float64

alpha = 0.75
pair_discrimination    0.705400
positive_mean          0.410849
negative_mean          0.192974
positive_median        0.357565
negative_median        0.048890
dtype: float64

alpha = 1.0
pair_discrimination    0.702900
positive_mean          0.488344
negative_mean          0.255813
positive_median        0.493438
negative_median        0.096434
dtype: float64


In [18]:
def build_role_embeddings(
    decks,
    *,
    alpha=0.5,
):
    """
    Represent each card by the average embedding of the
    rest of the decks in which that card appears.

    This measures similarity of deck context rather than
    direct card co-occurrence.
    """
    E = get_embedding(alpha)

    sums = np.zeros_like(E)
    counts = np.zeros(len(card_names), dtype=np.int64)

    for deck in decks:
        indices = [
            name_to_idx[name.lower()]
            for name in deck
            if name.lower() in name_to_idx
        ]

        if len(indices) < 2:
            continue

        # Sum of all cards in the deck.
        deck_sum = E[indices].sum(axis=0)

        for i in indices:
            # Context for card i = everything except i.
            context = deck_sum - E[i]

            norm = np.linalg.norm(context)

            if norm == 0:
                continue

            context /= norm

            sums[i] += context
            counts[i] += 1

    role_E = np.zeros_like(E)

    valid = counts > 0

    role_E[valid] = (
        sums[valid]
        / counts[valid, None]
    )

    norms = np.linalg.norm(
        role_E,
        axis=1,
        keepdims=True,
    )

    norms[norms == 0] = 1

    role_E /= norms

    return role_E, counts

In [19]:
role_E, role_counts = build_role_embeddings(
    decks,
    alpha=0.5,
)

In [20]:
def similar_roles(
    card_name,
    n=20,
    min_decks=5,
):
    i = name_to_idx[card_name.lower()]

    scores = role_E @ role_E[i]

    scores[i] = -np.inf
    scores[role_counts < min_decks] = -np.inf

    best = np.argsort(scores)[::-1][:n]

    return pd.DataFrame({
        "card": card_names[best],
        "role_similarity": scores[best],
        "deck_count": role_counts[best],
    })

In [21]:
similar_cards("Lightning Bolt")

,card,similarity
0,"Ragavan, Nimble Pilferer",0.974748
1,Chain Lightning,0.964131
2,"Inti, Seneschal of the Sun",0.960953
3,Mine Collapse,0.956462
4,Broadside Bombardiers,0.954346
5,"Laelia, the Blade Reforged",0.946855
6,Flame Slash,0.945833
7,"Magda, Brazen Outlaw",0.944229
8,Generous Plunderer,0.937683
9,Pyrokinesis,0.934741


In [22]:
similar_roles("Lightning Bolt")

,card,role_similarity,deck_count
0,Burst Lightning,0.999589,871
1,Flame Slash,0.999389,1161
2,Broadside Bombardiers,0.999291,1521
3,Chain Lightning,0.999180,1165
4,Fiery Confluence,0.998776,1154
5,Ghostfire Slice,0.998721,526
6,Galvanic Discharge,0.997964,1223
7,"Ragavan, Nimble Pilferer",0.997541,1195
8,Pyrokinesis,0.997496,875
9,"Chandra, Torch of Defiance",0.996190,644


In [30]:
similar_roles("Robber of the Rich")

,card,role_similarity,deck_count
0,Draconautics Engineer,0.999641,808
1,Detective's Phoenix,0.999517,548
2,Nova Hellkite,0.999497,641
3,Screaming Nemesis,0.999493,837
4,"Kari Zev, Skyship Raider",0.999224,678
5,Goblin Rabblemaster,0.999192,740
6,Arena of Glory,0.999172,942
7,"Kellan, Planar Trailblazer",0.999017,746
8,Hazoret the Fervent,0.998510,320
9,Death-Greeter's Champion,0.998486,1337
